In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import *
from fit_pv import *

In [ ]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [ ]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

In [ ]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [ ]:
folder_blos = '/home/ulyanov/data/solo/phi/2026/blos_/'
folder_vlos = '/home/ulyanov/data/solo/phi/2026/vlos_/'

In [ ]:
q_V = 299792458 / 6173.341

import fnmatch
from connect import bob

sftp = bob()

top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

folder = '/home/ulyanov/data/solo/phi/2026/'

for directory in dirs:
    if fnmatch.fnmatch(directory, '2026-04*'):
        for file in sorted(sftp.listdir(top_dir + directory))[:1]:
            if fnmatch.fnmatch(file, '*fdt-alam*C_*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                data, header = process(local_file,
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       #cavity_file=cavity_file,
                                       #flatfield_file=flat_file,
                                       #ghost_file=ghost_file,
                                       distortion_file=distortion_file,
                                       #_realign=True,
                                       _find_center=True,
                                       _demodulate=True,
                                       #_correct_fringes=True,
                                       #_correct_crosstalk=True,
                                       _calc_wavelengths=True,
                                       _mask=True,
                                       )

                data = rebin(data, 4, update_header=header)
                params = fit_line(data, header, correct_doppler=True, lam=1e-3, niter=10)[0]
                Vlos = params[0].clip(-0.5,0.5) * q_V

                stop

                #Blos, Vlos = classical_estimates(data, header)

                #file_blos = generate_filename(file, prefix='blos', folder=folder_blos)
                file_vlos = generate_filename(file, prefix='vlos', folder=folder_vlos)

                #clone_fits(local_file, file_blos, Blos, header)
                clone_fits(local_file, file_vlos, Vlos, header)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic', vmin=-0.1, vmax=0.1)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(params[1], 'inferno', vmin=0.15, vmax=0.2)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(-params[3] / params[2], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(params[2], 'inferno')
plt.tight_layout()

In [ ]:
print(np.nanmedian(params[0]))

In [ ]:
print(np.nanmedian(params[0]))